# Phases 2-4: PCA, Clustering & Validation

Dimensionality reduction, clustering, and cross-validation.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mega_env.config import load_config
from mega_env.processing import (
    prepare_covariate_matrix,
    standardize_and_pca,
    find_optimal_k,
    fit_clustering,
    aggregate_to_district_level,
    cross_validate_clustering,
    save_models
)
import joblib

config = load_config('../config/settings.yaml')

In [ ]:
# Load covariate table
covariate_df = pd.read_parquet('../data/processed/covariate_table.parquet')
print(f"Shape: {covariate_df.shape}")
covariate_df.head()

## Phase 2: PCA

In [ ]:
# Prepare matrix
df_clean, covariate_cols, district_col = prepare_covariate_matrix(covariate_df, config)
print(f"Clean shape: {df_clean.shape}")
print(f"Covariates: {len(covariate_cols)}")
print(f"District column: {district_col}")

In [ ]:
# Standardize and PCA
X_pca, pca, scaler, component_names = standardize_and_pca(df_clean[covariate_cols], config)

# Plot explained variance
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')

plt.subplot(1, 2, 2)
cumsum = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumsum)+1), cumsum, 'o-')
plt.axhline(y=config.pca_variance_threshold, color='r', linestyle='--', label=f'{config.pca_variance_threshold:.0%} threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Variance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA loadings heatmap
n_components = len(component_names)
loadings = pca.components_[:n_components, :]

plt.figure(figsize=(12, 8))
sns.heatmap(loadings, 
            xticklabels=covariate_cols, 
            yticklabels=component_names,
            cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.title('PCA Loadings (Component x Feature)')
plt.tight_layout()
plt.show()

## Phase 3: Clustering

In [ ]:
# Find optimal k
k_results = find_optimal_k(X_pca, config.clustering_k_range, config.clustering_random_state, config.clustering_n_init)

In [ ]:
# Plot silhouette scores
ks = list(k_results.keys())
silhouettes = [k_results[k]['silhouette'] for k in ks]
inertias = [k_results[k]['inertia'] for k in ks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, silhouettes, 'o-')
ax1.set_xlabel('k')
ax1.set_ylabel('Silhouette Score')
ax1.set_title('Silhouette Score vs k')
ax1.grid(True, alpha=0.3)

ax2.plot(ks, inertias, 'o-')
ax2.set_xlabel('k')
ax2.set_ylabel('Inertia')
ax2.set_title('Elbow Plot (Inertia vs k)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Select k (change based on plots above)
SELECTED_K = 5  # <-- Update after reviewing plots

cluster_model, labels = fit_clustering(
    X_pca, SELECTED_K, config.clustering_algorithm,
    config.clustering_random_state, config.clustering_n_init
)
print(f"Cluster distribution:")
for i in range(SELECTED_K):
    print(f"  Cluster {i}: {(labels == i).sum()} samples")

In [ ]:
# Visualize clusters in PC1-PC2 space
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.6, s=50)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'Clusters in PCA Space (k={SELECTED_K})')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Aggregate to district level
district_clusters = aggregate_to_district_level(df_clean, labels, district_col)
print(district_clusters[[district_col, 'cluster']].to_string(index=False))

In [ ]:
# Save models
save_models(scaler, pca, cluster_model, config)

# Save district clusters
district_clusters.to_csv('../data/processed/district_clusters.csv', index=False)
print("✓ Saved district clusters")

## Phase 4: Cross-Validation

In [ ]:
# Run cross-validation
cv_results = cross_validate_clustering(X_pca, df_clean, district_col, SELECTED_K, config)

In [ ]:
# Save validation results
import json
with open('../data/processed/validation_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)

print(f"Mean agreement: {cv_results['mean_agreement']:.3f} ± {cv_results['std_agreement']:.3f}")